# Anime Recommender — PySpark ALS
Java 17 is pre-installed in this container so no `--add-opens` workarounds are needed.

## 1. Spark Session

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('Spark UI: http://localhost:4040')

Spark version: 3.5.0
Spark UI: http://localhost:4040


## 2. Load Data

In [2]:
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviewsV2.csv', header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',    header=True, inferSchema=True)

print('reviews schema:')
reviews_data.printSchema()
print('animes schema:')
anime_data.printSchema()

reviews schema:
root
 |-- _c0: integer (nullable = true)
 |-- uid: integer (nullable = true)
 |-- profile: string (nullable = true)
 |-- anime_uid: integer (nullable = true)
 |-- score: integer (nullable = true)

animes schema:
root
 |-- _c0: integer (nullable = true)
 |-- anime_id: integer (nullable = true)
 |-- genre: string (nullable = true)



## 3. Prepare Final Data
Keep only rows with a valid score (score > 0) to avoid polluting the model with unrated entries.

In [4]:
from pyspark.ml.feature import StringIndexer
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col('user_id').cast('integer'),
    col('anime_uid').cast('integer'),
    col('score').cast('float')
).filter(col('score') > 0)  # drop unrated rows

print(f'Total rated reviews: {final_data.count():,}')
final_data.show(7)

Total rated reviews: 130,518
+-------+---------+-----+
|user_id|anime_uid|score|
+-------+---------+-----+
|     32|    34096|  8.0|
|   1104|    34599| 10.0|
|   1825|    28891|  7.0|
|   3796|     2904|  9.0|
|   9589|     4181| 10.0|
|   9872|     2904| 10.0|
|    554|    16664|  6.0|
+-------+---------+-----+
only showing top 7 rows



## 4. Train / Test Split

A naive `randomSplit` puts some users **only** in the test set — ALS has never seen them,
so every prediction gets dropped by `coldStartStrategy='drop'`, leaving an empty DataFrame
and the `Nothing has been added to this summarizer` error.

The fix: split **per user** by holding out 20 % of each user's ratings for testing.
This guarantees every test user also appears in training.

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rand, row_number, count

# Assign a random rank to each rating within every user
w = Window.partitionBy('user_id').orderBy(rand(seed=42))
ranked = final_data.withColumn('rn', row_number().over(w))

# Also compute how many ratings each user has
user_counts = final_data.groupBy('user_id').agg(count('*').alias('n'))
ranked = ranked.join(user_counts, on='user_id')

# Users with only 1 rating can't contribute to both sets — keep them in train only
train = ranked.filter((col('rn') / col('n') <= 0.8) | (col('n') == 1)).drop('rn', 'n')
test  = ranked.filter((col('rn') / col('n') >  0.8) & (col('n') >  1)).drop('rn', 'n')

print(f'Train size: {train.count():,}')
print(f'Test size:  {test.count():,}')

Train size: 101,756
Test size:  28,762


## 5. Train ALS Model

In [6]:
als = ALS(
    maxIter=10,
    regParam=0.1,
    rank=10,
    userCol='user_id',
    itemCol='anime_uid',
    ratingCol='score',
    coldStartStrategy='drop'  # safe now — all test users exist in train
)

model = als.fit(train)
print('Model trained.')

Model trained.


In [7]:
# Diagnosis cell — run this before anything else
print("Total rows:", final_data.count())
print("Rows with score > 0:", final_data.filter(col('score') > 0).count())
print("Distinct users:", final_data.select('user_id').distinct().count())

# Check score distribution
final_data.groupBy('score').count().orderBy('score').show(20)

Total rows: 130518
Rows with score > 0: 130518
Distinct users: 47885
+-----+-----+
|score|count|
+-----+-----+
|  1.0| 2916|
|  2.0| 3149|
|  3.0| 5726|
|  4.0| 5880|
|  5.0| 8548|
|  6.0|12446|
|  7.0|19064|
|  8.0|23727|
|  9.0|24543|
| 10.0|24518|
| 11.0|    1|
+-----+-----+



## 6. Evaluate — RMSE on Test Set

In [8]:
predictions = model.transform(test)

# Sanity check — should not be 0
print(f'Predictions count: {predictions.count():,}')

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='score',
    predictionCol='prediction'
)
rmse = evaluator.evaluate(predictions)
print(f'RMSE: {rmse:.4f}')
# RMSE is on a 1-10 scale; anything below ~1.5 is reasonable for anime ratings

Predictions count: 28,126
RMSE: 3.8601


## 7. Generate Recommendations

In [9]:
# Top 10 anime for every user
user_recs = model.recommendForAllUsers(10)
user_recs.show(5, truncate=False)

# Top 10 users for every anime
anime_recs = model.recommendForAllItems(10)
anime_recs.show(5, truncate=False)

+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                   |
+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|31     |[{34652, 15.166498}, {5494, 15.013835}, {4164, 15.013835}, {23697, 13.31176}, {16778, 13.193082}, {5194, 13.030551}, {7053, 12.682247}, {4017, 12.645955}, {1312, 12.366072}, {28157, 12.16998}]  |
|34     |[{1263, 12.354855}, {31471, 12.202639}, {25781, 12.181015}, {6505, 12.093636}, {2866, 11.99807}, {2987, 11.670709}, {38658, 11.479243}, {1846, 11.395295}, {10327, 11.15192

## 8. Recommend for a Specific User

In [11]:
# Pick any uid from the dataset — change this to explore different users
target_uid = final_data.select('user_id').first()[0]
print(f'Generating recommendations for uid: {target_uid}')

target_user = spark.createDataFrame([(target_uid,)], ['user_id'])
user_subset_recs = model.recommendForUserSubset(target_user, 10)

recs_flat = user_subset_recs.select(
    'user_id',
    explode('recommendations').alias('rec')
).select(
    'user_id',
    col('rec.anime_uid'),
    col('rec.rating')
)

recs_flat.show(truncate=False)

Generating recommendations for uid: 32
+-------+---------+---------+
|user_id|anime_uid|rating   |
+-------+---------+---------+
|32     |25923    |14.437494|
|32     |31189    |13.950393|
|32     |2799     |13.694846|
|32     |4639     |13.48478 |
|32     |7053     |12.57887 |
|32     |1724     |12.44269 |
|32     |15335    |12.408693|
|32     |2141     |12.214202|
|32     |2813     |12.165304|
|32     |1695     |12.130117|
+-------+---------+---------+



## 9. Join with Anime Metadata

In [12]:
# anime_data uses 'anime_id' but recs_flat uses 'anime_uid' — align the key
anime_lookup = anime_data.withColumnRenamed('anime_id', 'anime_uid')

readable_recs = recs_flat \
    .join(anime_lookup, on='anime_uid', how='left') \
    .select('user_id', 'anime_uid', 'genre', 'rating') \
    .orderBy('rating', ascending=False)

readable_recs.show(truncate=False)

+-------+---------+----------------------------------------------------------------------------+---------+
|user_id|anime_uid|genre                                                                       |rating   |
+-------+---------+----------------------------------------------------------------------------+---------+
|32     |25923    |['Hentai']                                                                  |14.437494|
|32     |31189    |['Hentai']                                                                  |13.950393|
|32     |2799     |['Adventure', 'Romance', 'Shoujo']                                          |13.694846|
|32     |4639     |['Sci-Fi', 'Comedy', 'Psychological', 'Drama']                              |13.48478 |
|32     |7053     |['Hentai']                                                                  |12.57887 |
|32     |1724     |['Adventure', 'Fantasy']                                                    |12.44269 |
|32     |15335    |['Action', 'Sci-Fi

## 10. (Optional) Hyperparameter Tuning
This trains 9 models × 3 folds = 27 fits. Skip if your dataset is large and you just want results.

In [13]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

param_grid = ParamGridBuilder() \
    .addGrid(als.rank,     [5, 10, 20]) \
    .addGrid(als.regParam, [0.01, 0.1, 1.0]) \
    .build()

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cv_model = cv.fit(train)
best_model = cv_model.bestModel
print(f'Best rank:     {best_model.rank}')
print(f'Best regParam: {best_model._java_obj.parent().getRegParam()}')

# Re-evaluate with the best model
best_preds = best_model.transform(test)
best_rmse  = evaluator.evaluate(best_preds)
print(f'Best model RMSE: {best_rmse:.4f}')

Best rank:     20
Best regParam: 1.0
Best model RMSE: 2.5750
